# ShockLink Tecplot 2D cut

This notebook reads the local BATSRUS Tecplot sample, extracts a planar cut, validates its arrays, and plots pressure with PyVista.

From the repository root, install and launch with:

```bash
pip install -e ".[notebook]"
jupyter lab examples/tecplot_2d_cut.ipynb
```

> **Memory note:** `data/3d.dat` is about 1.3 GB. The normalized grid and cut require additional memory.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import time

import numpy as np
import pyvista as pv


def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src/shocklink").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the ShockLink repository or examples directory")


ROOT = find_repository_root()
if importlib.util.find_spec("shocklink") is None:
    sys.path.insert(0, str(ROOT / "src"))

from shocklink.tecplot import get_2d_cut, plot_2d_cut, read_tecplot

pv.set_jupyter_backend("static")

In [ ]:
# Change these values to explore another file, plane, or color field.
DATA_PATH = ROOT / "data/3d.dat"
NORMAL = "z"
ORIGIN = (0.0, 0.0, 0.0)
SCALARS = "p"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Tecplot sample not found: {DATA_PATH}")

In [ ]:
started = time.perf_counter()
grid = read_tecplot(DATA_PATH)
load_seconds = time.perf_counter() - started

print(f"Loaded in {load_seconds:.3f} s")
print(f"Bounds: {tuple(float(value) for value in grid.bounds)}")
print(f"Point arrays: {list(grid.point_data.keys())}")
grid

In [ ]:
cut = get_2d_cut(grid, normal=NORMAL, origin=ORIGIN)

print(f"Cut points: {cut.n_points:,}")
print(f"Cut cells: {cut.n_cells:,}")
print(f"Cut bounds: {tuple(float(value) for value in cut.bounds)}")
print(f"Cut point arrays: {list(cut.point_data.keys())}")
cut

In [ ]:
required_arrays = {"P [nPa]", "B [nT]", "U [km/s]"}
assert required_arrays <= set(cut.point_data)

cut_normal = np.asarray(cut.field_data["shocklink_cut_normal"]).reshape(-1)
cut_origin = np.asarray(cut.field_data["shocklink_cut_origin"]).reshape(-1)
distances = (cut.points - cut_origin) @ cut_normal
np.testing.assert_allclose(distances, 0.0, atol=1e-6)

print("Cut validation passed")

In [ ]:
plotter = plot_2d_cut(cut, scalars=SCALARS, show=False)
plotter.show(jupyter_backend="static")